In [ ]:
from google.colab import drive
drive.mount('content')
%ls
%cd content/MyDrive/RecSys_Data

In [ ]:
%ls
%cd content/MyDrive/RecSys_Data

In [ ]:
%ls

In [ ]:
import csv
import pickle

In [ ]:
### data inspection

In [ ]:
articles = []
with (open("balanced_political_news_40k_v2.pkl", "rb")) as openfile:
    while True:
        try:
            articles.append(pickle.load(openfile))
        except EOFError:
            break

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(articles);

In [ ]:
df.head()

In [ ]:
sample = df[1741577].iloc[0]

In [ ]:
%pip install print_schema

In [ ]:
from print_schema import print_schema

In [ ]:
print_schema (sample)

In [ ]:
print(sample['kws_label'])

#cls_label actually matches topical_vector
print(sample['cls_label'])

In [ ]:
##feature vector - this was the feature for classification of topics - very sparse, in reality we may want a BERT model to generate embeddings
##topical vector - for user choice simulation

print(sample['topical_vector'])
print(sample['source_partisan_score'])

In [ ]:
print(sample['url'])

In [ ]:
#what info do we need for our BERT + adversarial autoencoder training?
#article_id, title, text, topical_vector

In [ ]:
#what we need for BERT
#article_id, title, text

temp = df.copy()

#what we need for adversarial autoencoder
#article_id, bert_embedding, source_partisan_score

In [ ]:
temp.head()

In [ ]:
print_schema(temp.iloc[0,1])

In [ ]:
def make_filter(temp, dict_keys):
  filter = pd.DataFrame(columns=dict_keys)

  for i in range(temp.shape[1]):
    row = temp.iloc[0, i]
    new = [row[key] for key in dict_keys]

    filter.loc[i] = new

  return filter

In [ ]:
bert_filter = make_filter(temp, ['article_id', 'title', 'text', 'source_partisan_score'])

In [ ]:
bert_filter.to_csv('bert_filter.csv')

In [ ]:
### train / test / val split - masks generated

In [ ]:
perm = make_filter(temp, ['article_id', 'source_partisan_score', 'cls_label'])

In [ ]:
topic_association = perm.copy()

In [ ]:
import numpy as np

In [ ]:
for row_id in range(topic_association.shape[0]):
  topics = np.array(topic_association.iloc[row_id]['cls_label'])
  r = np.random.randint(0, topics.size)

  topic_association.iloc[row_id, topic_association.columns.get_loc('cls_label')] = topics[r]




In [ ]:
topic_association.head()

In [ ]:
topic_association.to_csv('even_filter_base.csv')

In [ ]:
filter_base = pd.read_csv('even_filter_base.csv')

In [ ]:
df = filter_base
df['stratify_key'] = df.apply(lambda row: (row['source_partisan_score'], row['cls_label']), axis=1)

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# First split: train+val (90%) and test (10%)
train_val, test = train_test_split(
    df, test_size=0.1, stratify=df['stratify_key'], random_state=42
)

# Second split: train (80%) and val (10%)
train, val = train_test_split(
    train_val, test_size=0.1111, stratify=train_val['stratify_key'], random_state=42
)

In [ ]:
train.to_csv('training_mask.csv')
test.to_csv('testing_mask.csv')
val.to_csv('validation_mask.csv')

In [ ]:
print(train.shape)
print(test.shape)
print(val.shape)

In [ ]:
bert_set = pd.read_csv('bert_filter.csv')

In [ ]:
def filter_by_id(s, filtering_table, filter_col):
    filter = filtering_table[filter_col]
    new_set = s[s[filter_col].isin(filter)]
    ##print(new_set.shape)
    new_set.drop(columns=['Unnamed: 0'], inplace=True)
    ##print(new_set.head)
    return new_set

In [ ]:
train_set = filter_by_id(bert_set, train, 'article_id')
test_set = filter_by_id(bert_set, test, 'article_id')
val_set = filter_by_id(bert_set, val, 'article_id')

In [ ]:
train_set.head()

In [ ]:
train_set = train_set.sample(frac=1)
train_set = train_set.sample(frac=1)
train_set = train_set.sample(frac=1)
train_set = train_set.sample(frac=1)

In [ ]:
test_set = test_set.sample(frac=1)
test_set = test_set.sample(frac=1)
test_set = test_set.sample(frac=1)
test_set = test_set.sample(frac=1)

In [ ]:
val_set = val_set.sample(frac=1)
val_set = val_set.sample(frac=1)
val_set = val_set.sample(frac=1)
val_set = val_set.sample(frac=1)

In [ ]:
train_set.to_csv('bert_training_set.csv')
test_set.to_csv('bert_testing_set.csv')
val_set.to_csv('bert_validation_set.csv')

In [ ]:
######### Dataset to generate bert embeddings over

In [ ]:
# bert_filter = make_filter(temp, ['article_id', 'title', 'text'])

In [ ]:
# bert_filter.head()

In [ ]:
# bert_filter.describe()

In [ ]:
# bert_filter.to_csv('bert_filter.csv')